# Transfermarkt scraper — top 5 leagues, 2017–2025

LaLiga, Premier League, Bundesliga, Serie A and Ligue 1 — the list, with each
league's URL, country and clubs per season, lives in `src/data/leagues.py`.

1. **Rosters** — league page per (league, season) → team pages → one row per
   (season, club, player), with league, country, birth date and photo.
   `data/_rosters.parquet`, saved after every league-season.
2. **Profiles** — per *player*: nationality, height, foot, position.
   `data/_enrichment_checkpoint.csv`.
3. **Value histories** — per *player*, every market-value snapshot Transfermarkt
   has, in **EUR for every league** (the API returns euros for the Premier League
   too; anything else stops the run). `data/_value_histories.csv`.
4. **Season values** — each row's market value is the snapshot at the **end** of
   its season (June of year+1). Age is exact, from the birth date.
5. **Fill** — two-sided gaps in market value; ages without a birth date.

Why the end of the season: Understat stats cover the whole season (August to
May). A mid-season (December) value would be "explained" by goals scored after
it was set — the model would be learning from the future.

Every request goes through `fetch()` (retry, backoff, pacing), and every stage
checkpoints to disk, so an interrupted run resumes instead of starting over.
Re-running the whole notebook only fetches what is missing.

In [ ]:
import csv
import os
import random
import re
import sys
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


def find_project_root(marker: str = "src/features/engineer.py") -> Path:
    """Walk up from the working directory until the repo root turns up."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"could not find {marker} at or above {here}")


BASE_PATH = Path(os.environ.get("TRANSFER_EDGE_ROOT") or find_project_root())
sys.path.insert(0, str(BASE_PATH))
from src.data.leagues import LEAGUES, clubs_in_season, tm_league_url  # noqa: E402

DATA_DIR = BASE_PATH / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ROSTER_PATH = DATA_DIR / "_rosters.parquet"              # stage 1 output
CHECKPOINT_PATH = DATA_DIR / "_enrichment_checkpoint.csv"  # stage 2 progress
PANEL_PATH = DATA_DIR / "players_panel.parquet"          # final output

LEAGUE_CODES = list(LEAGUES)      # ES1, GB1, L1, IT1, FR1 — clubs per season asserted per league
YEARS = list(range(2017, 2026))   # saison_id 2017 == the 2017-18 season
REQUEST_DELAY = 0.2              # seconds between successful requests
MAX_RETRIES = 4

VALUE_API = "https://tmapi-alpha.transfermarkt.technology/player/{player_id}/market-value-history"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"
    )
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

print(f"project root: {BASE_PATH}")
print(f"leagues: {', '.join(f'{c} ({LEAGUES[c].name})' for c in LEAGUE_CODES)}")

In [ ]:
class ScrapeError(RuntimeError):
    """A URL that still failed after every retry."""


def fetch(url: str, *, as_json: bool = False, timeout: int = 30):
    """GET with retry, exponential backoff and a pause between calls.

    Raises ScrapeError instead of returning None, so being rate-limited can
    never be mistaken for a player whose profile genuinely omits a field.
    """
    delay, last_error = 0.2, None
    for attempt in range(MAX_RETRIES):
        try:
            resp = SESSION.get(url, timeout=timeout)
            resp.raise_for_status()
            payload = resp.json() if as_json else BeautifulSoup(resp.content, "html.parser")
            time.sleep(REQUEST_DELAY)
            return payload
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt == MAX_RETRIES - 1:
                break
            time.sleep(delay + random.uniform(0, 0.5))  # jitter so retries desynchronise
            delay *= 2
    raise ScrapeError(f"{url} failed after {MAX_RETRIES} attempts: {last_error}")

## 1. Rosters

Club and player links are selected by their URL shape (`/startseite/verein/`,
`/profil/spieler/`) rather than by position in the table, so an extra promo row
on the page cannot shift the results the way a `[:-5][:20]` slice would.

In [ ]:
def season_team_links(league: str, year: int) -> list[tuple[str, str]]:
    """Return [(club_name, club_url)] for one league-season, validated against leagues.py."""
    url = tm_league_url(league, year)
    soup = fetch(url)

    teams, seen = [], set()
    for a in soup.select("table.items td.hauptlink a[href*='/startseite/verein/']"):
        name = a.get_text(strip=True)
        if not name or name in seen:
            continue
        seen.add(name)
        teams.append((name, urljoin(url, a["href"])))

    expected = clubs_in_season(league, year)
    if len(teams) != expected:
        raise ScrapeError(
            f"{league} {year}: expected {expected} clubs, parsed {len(teams)} — "
            f"Transfermarkt's layout has probably changed. Got: {[t[0] for t in teams]}"
        )
    return teams


def parse_birth_date(text):
    """'Apr 30, 1992 (34)' -> '1992-04-30'. None if there is no date in it."""
    match = re.search(r"([A-Z][a-z]{2} \d{1,2}, \d{4})", text or "")
    if not match:
        return None
    try:
        return datetime.strptime(match.group(1), "%b %d, %Y").date().isoformat()
    except ValueError:
        return None


def portrait_url(img):
    """Full-size portrait URL from a squad-table <img>, or None.

    The table lazy-loads: the real URL sits in data-src and src is a 1x1 gif.
    Squad tables serve the 'medium' size; the profile header uses 'header',
    which is the same photo, larger. Transfermarkt's grey silhouette for
    players without a photo is dropped rather than stored.
    """
    if img is None:
        return None
    url = img.get("data-src") or img.get("src") or ""
    if not url.startswith("http") or "/portrait/" not in url or "default" in url:
        return None
    return url.replace("/portrait/medium/", "/portrait/header/")


def parse_team_page(soup, team_url: str) -> list[dict]:
    """One dict per player row in a squad table. Deduplicated by player_id."""
    players, seen = [], set()
    for tr in soup.select("table.items > tbody > tr"):
        a = tr.select_one("td.hauptlink a[href*='/profil/spieler/']")
        if a is None:
            continue
        href = urljoin(team_url, a["href"])
        match = re.search(r"/profil/spieler/(\d+)", href)
        if not match or match.group(1) in seen:
            continue
        seen.add(match.group(1))
        birth = next((parse_birth_date(td.get_text(" ", strip=True))
                      for td in tr.find_all("td", recursive=False)
                      if parse_birth_date(td.get_text(" ", strip=True))), None)
        players.append({
            "player_id": match.group(1),
            "player_name": a.get_text(strip=True),
            "link_html": href,
            "birth_date": birth,
            "image_url": portrait_url(tr.select_one("img.bilderrahmen-fixed")),
        })
    return players


LEAGUE_COLS = ["league_code", "league", "league_country"]


def with_league(df: pd.DataFrame, code: str) -> pd.DataFrame:
    """Stamp a league's code, display name and country on every row."""
    league = LEAGUES[code]
    return df.assign(league_code=code, league=league.name, league_country=league.country)


def scrape_league_season(league: str, year: int) -> pd.DataFrame:
    """One row per (club, player) in one league-season, with birth date and photo URL."""
    rows = []
    for club, team_url in season_team_links(league, year):
        for p in parse_team_page(fetch(team_url), team_url):
            rows.append({"year": year, "club": club, **p})
    return with_league(pd.DataFrame(rows), league)


def scrape_missing_rosters(roster: pd.DataFrame, leagues=LEAGUE_CODES, years=YEARS) -> pd.DataFrame:
    """Scrape every (league, season) not in `roster` yet; save after each one."""
    have = set(zip(roster["league_code"], roster["year"].astype(int))) if len(roster) else set()
    todo = [(lg, y) for lg in leagues for y in years if (lg, y) not in have]
    print(f"{len(have)} league-seasons cached, {len(todo)} to scrape")
    for league, year in todo:
        season = scrape_league_season(league, year)
        roster = pd.concat([roster, season], ignore_index=True)
        roster.to_parquet(ROSTER_PATH, index=False)
        print(f"  {league} {year}: {season.club.nunique()} clubs, {len(season)} players")
    return roster


def add_roster_details(roster: pd.DataFrame) -> pd.DataFrame:
    """Add birth_date / image_url to a roster cached before they were scraped.

    Re-reads only the team pages, never the player profiles, and leaves every
    existing roster row as it was.
    """
    found = []
    for league, year in sorted(set(zip(roster["league_code"], roster["year"].astype(int)))):
        for club, team_url in season_team_links(league, int(year)):
            for p in parse_team_page(fetch(team_url), team_url):
                found.append({"year": int(year), "player_id": p["player_id"],
                              "birth_date": p["birth_date"], "image_url": p["image_url"]})
        print(f"{league} {year}: details read")
    details = pd.DataFrame(found).drop_duplicates(subset=["year", "player_id"])
    return (roster.drop(columns=["birth_date", "image_url"], errors="ignore")
            .astype({"year": int, "player_id": str})
            .merge(details, on=["year", "player_id"], how="left"))

In [ ]:
if ROSTER_PATH.exists():
    roster = pd.read_parquet(ROSTER_PATH)
    print(f"reusing {ROSTER_PATH.name}: {roster.shape}")
    if "league_code" not in roster.columns:
        # Cached by the La Liga-only version of this notebook.
        roster = with_league(roster, "ES1")
        roster.to_parquet(ROSTER_PATH, index=False)
        print("  stamped the cached rows as ES1 (LaLiga)")
else:
    roster = pd.DataFrame()

roster = scrape_missing_rosters(roster)

# Rosters cached before photos / birth dates were scraped get them added here —
# team pages only, so the (expensive) enrichment checkpoint stays valid.
if not {"birth_date", "image_url"} <= set(roster.columns):
    roster = add_roster_details(roster)
    roster.to_parquet(ROSTER_PATH, index=False)
    print(f"added birth_date / image_url -> {ROSTER_PATH.name}: {roster.shape}")

roster["year"] = roster["year"].astype(int)
roster["player_id"] = roster["player_id"].astype(str)
print(f"\nroster: {roster.shape}, {roster.player_id.nunique()} players")
print(roster.groupby("league").agg(seasons=("year", "nunique"), rows=("player_id", "size"),
                                   players=("player_id", "nunique")).to_string())
print(f"with birth date: {roster.birth_date.notna().mean():.0%}   "
      f"with photo: {roster.image_url.notna().mean():.0%}")
roster.head()

## 2. Profiles

One request per **player**, not per season: the profile URL is the same in every
season and the page shows his current details (checked on the La Liga data: the
fields agreed across seasons for 2,585 of 2,586 players), so asking once per
season fetched the same page several times.

Each completed player is appended to `_enrichment_checkpoint.csv` and flushed,
so a crash or a ban costs only the player in flight; players that fail every
retry are not checkpointed, so re-running the cell retries exactly those.

Market value and age are no longer taken here — see sections 3 and 4.

In [ ]:
def info_table_value(soup, label: str):
    """Read one value out of Transfermarkt's info table by its label."""
    for span in soup.select("span.info-table__content--regular"):
        if span.get_text(strip=True) == label:
            sibling = span.find_next_sibling("span", class_="info-table__content--bold")
            if sibling:
                return sibling.get_text(strip=True)
    return None


def parse_height_m(text: str):
    """Parse a height string to metres.

    transfermarkt.us renders height as "6 ft 2 in" (imperial), not the "1,87 m"
    (metric, comma decimal) the .com/.de domains use. Handle both, since which
    one you get can depend on the domain and may change again.
    """
    if not text:
        return None
    text = text.strip()

    imperial = re.match(r"(\d+)\s*ft\s*(\d+)\s*in", text)
    if imperial:
        feet, inches = int(imperial.group(1)), int(imperial.group(2))
        return round((feet * 12 + inches) * 0.0254, 2)

    metric = re.match(r"([\d,.]+)\s*m", text)
    if metric:
        try:
            return float(metric.group(1).replace(",", "."))
        except ValueError:
            return None

    return None


def parse_profile(soup) -> dict:
    """Static attributes from a profile page; None where Transfermarkt omits one.

    Every field here is genuinely optional. Transport failures raise out of
    fetch() instead, which is what keeps "not listed" and "we got blocked"
    distinguishable.
    """
    nationality = soup.select_one('span[itemprop="nationality"]')

    height = None
    height_tag = soup.select_one('span[itemprop="height"]')
    if height_tag:
        height = parse_height_m(height_tag.get_text(strip=True))

    return {
        "nationality": nationality.get_text(strip=True) if nationality else None,
        "height": height,
        "foot": info_table_value(soup, "Foot:"),
        "position": info_table_value(soup, "Position:"),
    }

In [ ]:
HISTORY_PATH = DATA_DIR / "_value_histories.csv"   # stage 3 progress
HISTORY_COLS = ["player_id", "date", "market_value_eur", "club_id", "currency"]


class CurrencyError(RuntimeError):
    """The value API answered in something other than EUR. Stops the run on purpose."""


def _add_currency_column():
    """Histories cached before the currency check get an empty `currency` column.

    Empty means "fetched before the check", not "unknown currency": those rows
    are LaLiga players, and the API returns EUR for every league.
    """
    if not HISTORY_PATH.exists():
        return
    with HISTORY_PATH.open() as fh:
        header = fh.readline().strip().split(",")
    if "currency" not in header:
        old = pd.read_csv(HISTORY_PATH, dtype=str)
        old.reindex(columns=HISTORY_COLS).to_csv(HISTORY_PATH, index=False)
        print(f"added an empty currency column to {len(old):,} cached rows")


def fetch_value_histories(player_ids) -> pd.DataFrame:
    """Every market-value snapshot for every player — one request per player.

    Checkpointed like the profiles: a player whose history is empty still gets
    a marker row (no date), so a re-run doesn't ask for him again. Every snapshot
    must be in EUR; anything else raises CurrencyError before it is written.
    """
    _add_currency_column()
    done = set()
    if HISTORY_PATH.exists():
        done = set(pd.read_csv(HISTORY_PATH, dtype={"player_id": str}, usecols=["player_id"]).player_id)
    todo = [p for p in dict.fromkeys(map(str, player_ids)) if p not in done]
    print(f"{len(done)} players cached, {len(todo)} to fetch")

    failures = []
    write_header = not HISTORY_PATH.exists()
    with HISTORY_PATH.open("a", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=HISTORY_COLS)
        if write_header:
            writer.writeheader()
        for n, pid in enumerate(todo, 1):
            try:
                payload = fetch(VALUE_API.format(player_id=pid), as_json=True) or {}
            except ScrapeError as exc:
                failures.append(pid)
                print(f"  ! {pid}: {exc}")
                continue                      # not checkpointed, so a re-run retries it
            snapshots = []
            for h in payload.get("data", {}).get("history") or []:
                mv = h.get("marketValue") or {}
                if not mv.get("determined"):
                    continue
                if mv.get("currency") != "EUR":
                    raise CurrencyError(f"player {pid}: value in {mv.get('currency')!r}, not EUR — {mv}")
                snapshots.append({"player_id": pid, "date": mv["determined"],
                                  "market_value_eur": mv["value"], "club_id": h.get("clubId"),
                                  "currency": "EUR"})
            writer.writerows(snapshots or [{"player_id": pid}])
            fh.flush()
            if n % 200 == 0:
                print(f"  {n}/{len(todo)}")

    if failures:
        print(f"\n{len(failures)} players failed — re-run this cell to retry them")
    hist = pd.read_csv(HISTORY_PATH, dtype={"player_id": str, "club_id": str, "currency": str})
    hist["date"] = pd.to_datetime(hist["date"], errors="coerce")
    return hist.dropna(subset=["date"])


def value_at_season_end(hist: pd.DataFrame, year: int):
    """The snapshot closest to the end of season `year` (which ends in year+1).

    June of year+1 first — Transfermarkt's big post-season update — then the
    last one in May, then the first one in July. Returns (value, date), or
    (None, None) when none of those months has a snapshot.
    """
    if hist is None or hist.empty:
        return None, None
    end = year + 1
    for month, which in ((6, "last"), (5, "last"), (7, "first")):
        hits = hist[(hist["date"].dt.year == end) & (hist["date"].dt.month == month)]
        if hits.empty:
            continue
        row = hits.sort_values("date").iloc[-1 if which == "last" else 0]
        return float(row["market_value_eur"]), row["date"]
    return None, None


def age_at_season_end(birth_date: pd.Series, year: pd.Series) -> pd.Series:
    """Exact age on 30 June of year+1 — the date the season's value is taken."""
    born = pd.to_datetime(birth_date, errors="coerce")
    not_yet = (born.dt.month > 6).astype(float)   # birthday after 30 June: not had it yet
    return (year.astype(int) + 1) - born.dt.year - not_yet

In [ ]:
FIELDS = ["year", "player_id", "nationality", "height", "foot", "position"]
PROFILE_FIELDS = FIELDS[2:]


def enrich(roster: pd.DataFrame) -> pd.DataFrame:
    """Fetch each player's profile once, checkpointing as it goes, and join it to every season."""
    done, header = set(), FIELDS
    if CHECKPOINT_PATH.exists():
        prior = pd.read_csv(CHECKPOINT_PATH, dtype={"player_id": str})
        done = set(prior.player_id)
        header = list(prior.columns)   # older checkpoints carry extra columns; keep appends aligned
        print(f"resuming — {len(done)} players already fetched")

    # The latest season's row per player: same profile URL in every season.
    players = (roster.sort_values("year").drop_duplicates("player_id", keep="last"))
    todo = [r for r in players.itertuples(index=False) if str(r.player_id) not in done]
    print(f"{len(todo)} players to fetch")

    failures = []
    with CHECKPOINT_PATH.open("a", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=header, extrasaction="ignore")
        if not done:
            writer.writeheader()

        for n, r in enumerate(todo, 1):
            record = {"year": int(r.year), "player_id": str(r.player_id)}
            try:
                record.update(parse_profile(fetch(r.link_html)))
            except ScrapeError as exc:
                failures.append((r.year, r.player_id, r.player_name))
                print(f"  ! {r.player_name} ({r.year}): {exc}")
                continue                      # not checkpointed, so a re-run retries it
            writer.writerow(record)
            fh.flush()
            if n % 100 == 0:
                print(f"  {n}/{len(todo)}")

    if failures:
        print(f"\n{len(failures)} players failed — re-run this cell to retry them")

    # One profile per player (the most recently fetched), joined to all his seasons.
    # A December market_value / age from older runs is ignored on purpose.
    fetched = (
        pd.read_csv(CHECKPOINT_PATH, dtype={"player_id": str})
        .drop_duplicates(subset=["player_id"], keep="last")
        [["player_id", *PROFILE_FIELDS]]
    )
    return roster.merge(fetched, on="player_id", how="left")


panel = enrich(roster)
print(f"\nenriched panel: {panel.shape}   with a position: {panel.position.notna().mean():.0%}")
panel.head()

## 3. Value histories

One request per player (not per season): the API returns the player's whole
history in one go, so asking again for every season he played was wasted work.

In [ ]:
histories = fetch_value_histories(roster["player_id"].unique())
print(f"\n{histories.player_id.nunique()} players with a history, {len(histories):,} snapshots")
histories.head()

## 4. Season values and ages

Each row takes the snapshot at the end of its season (`value_date` records
which one). Age is computed from the birth date on the same date, so it is exact
and consistent with the value.

In [ ]:
by_player = {pid: g for pid, g in histories.groupby("player_id")}
season_values = [value_at_season_end(by_player.get(str(r.player_id)), int(r.year))
                 for r in panel.itertuples(index=False)]
panel["market_value"] = [v for v, _ in season_values]
panel["value_date"] = [dt for _, dt in season_values]
panel["age"] = age_at_season_end(panel["birth_date"], panel["year"])

print("rows with an end-of-season value, by league:")
print(panel.groupby("league").market_value.apply(lambda s: f"{s.notna().mean():.0%}").to_string())
print("month the value was taken from:")
print(pd.to_datetime(panel.value_date).dt.month.value_counts().rename({5: "May", 6: "June", 7: "July"}).to_string())
print(f"rows with an age: {panel.age.notna().mean():.0%}")

## 5. Fill missing market values and ages

**Market value.** A gap is filled with the **mean of the nearest season before
and the nearest season after it in which he has a listed value** — and only
when both sides exist. A gap at the start or end of his record stays empty (and
the row drops out at `build_features`): copying a neighbour's value there would
put a later valuation into an earlier season.

Filled rows are flagged in `market_value_imputed` and are **never used by the
model** — not as training targets, not as test rows, not as lags. The midpoint
is built from the *next* season's value, so using it anywhere in the model would
leak the future. They exist only so the app's value history has no holes.

**Age.** Already exact from the birth date (section 4). The few rows without a
birth date take it from a neighbouring season plus the year difference.

In [ ]:
def fill_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Fill market_value (two-sided midpoint only) and ages with no birth date."""
    df = df.copy()
    df["year"] = df["year"].astype(int)
    for col in ("age", "market_value"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.sort_values(["player_id", "year"]).reset_index(drop=True)

    # Market value: midpoint of the nearest listed season on each side. NaN
    # unless both exist — deliberately no one-sided fallback (see markdown).
    value_missing = df["market_value"].isna()
    by_player = df.groupby("player_id")
    midpoint = (by_player["market_value"].ffill() + by_player["market_value"].bfill()) / 2
    df["market_value"] = df["market_value"].fillna(midpoint)
    df["market_value_imputed"] = value_missing & df["market_value"].notna()

    # Age: a neighbouring season's age plus the year difference (exact too).
    age_missing = df["age"].isna()
    df["_known_age"] = df["age"]
    df["_known_year"] = df["year"].where(df["age"].notna())
    by_player = df.groupby("player_id")
    from_prev = by_player["_known_age"].ffill() + (df["year"] - by_player["_known_year"].ffill())
    from_next = by_player["_known_age"].bfill() - (by_player["_known_year"].bfill() - df["year"])
    df["age"] = df["age"].fillna(from_prev).fillna(from_next)
    df["age_imputed"] = age_missing & df["age"].notna()

    return df.drop(columns=["_known_age", "_known_year"])


n_missing_value = int(panel["market_value"].isna().sum())
n_missing_age = int(panel["age"].isna().sum())
panel = fill_missing_values(panel)

print(f"market values missing : {n_missing_value}")
print(f"  filled (two-sided)  : {int(panel.market_value_imputed.sum())}  (display only, never modelled)")
print(f"  left empty          : {int(panel.market_value.isna().sum())}")
print(f"ages missing          : {n_missing_age}")
print(f"  still missing       : {int(panel.age.isna().sum())}")

In [ ]:
# League columns first, after the season, so the panel reads naturally.
first = ["year", *LEAGUE_COLS, "club"]
panel = panel[first + [c for c in panel.columns if c not in first]]

panel.to_parquet(PANEL_PATH, index=False)
print(f"saved {PANEL_PATH.name}: {panel.shape}")
print(f"columns: {list(panel.columns)}\n")
print(panel.groupby("league").agg(
    rows=("player_id", "size"), players=("player_id", "nunique"),
    median_value_eur=("market_value", "median"),
).to_string())
panel.head()